In [4]:
# 03_eda_analisis_exploratorio.ipynb
# Analisis Exploratorio de Datos (EDA) para clustering
# Objetivo: entender la estructura, distribucion y relaciones de las variables acusticas

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew

# ============================================
# CONFIGURACION
# ============================================
RUTA_FEATURES = "./../data/processed/spotify_features_clean.csv"
RUTA_FIGURAS = "./../docs/figures/"

os.makedirs(RUTA_FIGURAS, exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("Set2")
plt.rcParams['figure.figsize'] = (10, 6)

print("="*70)
print("EDA PARA CLUSTERING DE CANCIONES DE SPOTIFY")
print("="*70)

# ============================================
# 1. CARGA DE DATOS
# ============================================
print("\n1. CARGANDO DATOS PREPROCESADOS...")
df = pd.read_csv(RUTA_FEATURES)
print(f"   Registros: {df.shape[0]:,}")
print(f"   Variables: {df.shape[1]}")
print(f"   Columnas: {df.columns.tolist()}")

# ============================================
# 2. VISION GENERAL
# ============================================
print("\n2. VISION GENERAL DE LOS DATOS...")
print(f"   - Valores nulos: {df.isnull().sum().sum()}")
print(f"   - Duplicados: {df.duplicated().sum():,}")
print(f"   - Memoria: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# ============================================
# 3. ESTADISTICOS DESCRIPTIVOS
# ============================================
print("\n3. ESTADISTICOS DESCRIPTIVOS...")
estadisticos = df.describe()
print(estadisticos.round(4))

print("\n   Rangos por variable:")
for col in df.columns:
    print(f"   {col}: [{df[col].min():.4f}, {df[col].max():.4f}]")

# ============================================
# 4. MATRIZ DE CORRELACION
# ============================================
print("\n4. ANALISIS DE CORRELACIONES...")

corr_matrix = df.corr()

corr_alta = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        val = corr_matrix.iloc[i, j]
        if abs(val) > 0.6:
            corr_alta.append((corr_matrix.columns[i], corr_matrix.columns[j], val))

if corr_alta:
    print("\n   Correlaciones fuertes (>0.6 o <-0.6):")
    for v1, v2, val in corr_alta:
        print(f"   - {v1} vs {v2}: {val:.3f}")
else:
    print("\n   No se encontraron correlaciones fuertes.")

# Heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f',
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Matriz de correlacion - Variables acusticas', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(RUTA_FIGURAS, "matriz_correlacion.pdf"), format='pdf')
plt.savefig(os.path.join(RUTA_FIGURAS, "matriz_correlacion.png"), dpi=300)
plt.close()
print("\n   Matriz de correlacion guardada en docs/figures/")

# ============================================
# 5. HISTOGRAMAS
# ============================================
print("\n5. ANALISIS DE DISTRIBUCIONES...")

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(df.columns):
    axes[i].hist(df[col], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel('Valor')
    axes[i].set_ylabel('Frecuencia')
    axes[i].axvline(df[col].mean(), color='red', linestyle='--', linewidth=1, label='Media')

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribucion de las variables acusticas', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(RUTA_FIGURAS, "histogramas_variables.pdf"), format='pdf')
plt.savefig(os.path.join(RUTA_FIGURAS, "histogramas_variables.png"), dpi=300)
plt.close()
print("   Histogramas guardados en docs/figures/")

# ============================================
# 6. BOXPLOTS
# ============================================
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(df.columns):
    axes[i].boxplot(df[col], vert=True, patch_artist=True,
                    boxprops=dict(facecolor='steelblue'))
    axes[i].set_title(col, fontsize=10)
    axes[i].set_ylabel('Valor')

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Deteccion de outliers (boxplots)', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(RUTA_FIGURAS, "boxplots_variables.pdf"), format='pdf')
plt.savefig(os.path.join(RUTA_FIGURAS, "boxplots_variables.png"), dpi=300)
plt.close()
print("   Boxplots guardados en docs/figures/")

# ============================================
# 7. ASIMETRIA
# ============================================
print("\n6. ANALISIS DE ASIMETRIA...")

print("\n   Coeficiente de asimetria (skewness) por variable:")
for col in df.columns:
    sk = skew(df[col])
    if abs(sk) > 1:
        print(f"   {col}: {sk:.3f} (alta asimetria)")
    elif abs(sk) > 0.5:
        print(f"   {col}: {sk:.3f} (asimetria moderada)")
    else:
        print(f"   {col}: {sk:.3f} (aproximadamente simetrica)")

# ============================================
# 8. ANALISIS DE MODO (sin warning)
# ============================================
print("\n7. ANALISIS DE MODO (tonalidad mayor/menor)...")

if 'mode' in df.columns:
    mode_counts = df['mode'].value_counts()
    mode_pct = df['mode'].value_counts(normalize=True) * 100
    
    print(f"\n   Canciones en modo mayor (1): {mode_counts.get(1, 0):,} ({mode_pct.get(1, 0):.1f}%)")
    print(f"   Canciones en modo menor (0): {mode_counts.get(0, 0):,} ({mode_pct.get(0, 0):.1f}%)")
    
    # Grafico de barras (corregido sin warning)
    plt.figure(figsize=(6, 4))
    sns.countplot(data=df, x='mode', hue='mode', palette='Set2', legend=False)
    plt.title('Distribucion de canciones por modo (mayor vs menor)', fontsize=12)
    plt.xlabel('Modo (1=Mayor, 0=Menor)')
    plt.ylabel('Cantidad de canciones')
    plt.tight_layout()
    plt.savefig(os.path.join(RUTA_FIGURAS, "distribucion_modo.pdf"), format='pdf')
    plt.savefig(os.path.join(RUTA_FIGURAS, "distribucion_modo.png"), dpi=300)
    plt.close()
    print("   Grafico de modo guardado en docs/figures/")

# ============================================
# 9. CONCLUSIONES
# ============================================
print("\n" + "="*70)
print("CONCLUSIONES DEL EDA PARA CLUSTERING")
print("="*70)

print(f"""
A. ESTRUCTURA DE LOS DATOS:
   - {df.shape[0]:,} registros, {df.shape[1]} variables numericas continuas
   - Sin valores nulos
   - Las variables estan en rangos compatibles

B. CORRELACIONES RELEVANTES:
""")

if corr_alta:
    for v1, v2, val in corr_alta:
        print(f"   - {v1} vs {v2}: {val:.3f}")
else:
    print("   - No hay correlaciones fuertes (>0.6)")

print("""
C. DISTRIBUCIONES Y ASIMETRIA:
   - instrumentalness, speechiness, liveness, loudness, duration_ms: alta asimetria
   - Esto justifica la transformacion logaritmica aplicada en preprocesamiento

D. IMPLICACIONES PARA CLUSTERING:
   - Las variables tienen escalas diferentes -> se necesita estandarizacion
   - No hay correlaciones extremadamente altas -> no es necesario eliminar variables
   - El dataset es grande (2M+ registros) -> se recomienda muestreo

E. PREGUNTAS PARA EL CLUSTERING:
   1. Cuantos grupos naturales de canciones existen?
   2. Como se agrupan segun danceability, energy y valence?
   3. Las canciones explicitas se concentran en clusters especificos?
""")

# Exportar estadisticos
estadisticos.to_csv(os.path.join(RUTA_FIGURAS, "estadisticos_descriptivos.csv"))
print("\nEstadisticos descriptivos guardados en docs/figures/estadisticos_descriptivos.csv")

print("\n" + "="*70)
print("EDA COMPLETADO. LISTO PARA EL NOTEBOOK DE CLUSTERING.")
print("="*70)

EDA PARA CLUSTERING DE CANCIONES DE SPOTIFY

1. CARGANDO DATOS PREPROCESADOS...
   Registros: 2,110,286
   Variables: 11
   Columnas: ['danceability', 'energy', 'valence', 'acousticness', 'instrumentalness', 'liveness', 'speechiness', 'tempo', 'loudness', 'duration_ms', 'mode']

2. VISION GENERAL DE LOS DATOS...
   - Valores nulos: 0
   - Duplicados: 2,085,835
   - Memoria: 177.10 MB

3. ESTADISTICOS DESCRIPTIVOS...
       danceability        energy       valence  acousticness  \
count  2.110286e+06  2.110286e+06  2.110286e+06  2.110286e+06   
mean   6.759000e-01  6.488000e-01  5.463000e-01  2.749000e-01   
std    1.440000e-01  1.689000e-01  2.312000e-01  2.509000e-01   
min    6.570000e-02  0.000000e+00  0.000000e+00  0.000000e+00   
25%    5.800000e-01  5.520000e-01  3.700000e-01  6.670000e-02   
50%    7.000000e-01  6.680000e-01  5.480000e-01  1.910000e-01   
75%    7.800000e-01  7.670000e-01  7.330000e-01  4.370000e-01   
max    9.880000e-01  9.980000e-01  9.920000e-01  9.960000e-0